In [ ]:
OUTPUT_PDF_DIR = (
    r"" # Path for output directory
)

import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from cycler import cycler
os.makedirs(OUTPUT_PDF_DIR, exist_ok=True)

SET1_6 = ['#e41a1c', '#377eb8', '#4daf4a',
          '#984ea3', '#ff7f00', '#a65628']

BASE_RC = {
    'figure.figsize': (5, 3),
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'legend.fontsize': 8,
    'lines.linewidth': 1.2,
    'axes.prop_cycle': cycler('color', SET1_6),
}

def _add_grid(ax):
    ax.grid(which='major', axis='x',
            linestyle='dashdot', linewidth=0.4, color="#AEAEAE")
    ax.grid(which='minor', axis='x',
            linestyle='dotted',  linewidth=0.2, color="#AEAEAE")

def _common_format(ax, xlabel, ylabel, xmax):
    """Apply axes labels, limits, ticks and grid in a single call."""
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, xmax)

    maj = max(10, int(xmax // 8))
    min_ = max(5,  int(xmax // 16))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(maj))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(min_))
    _add_grid(ax)
    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 1.22),
        ncol=3,
        frameon=False
    )

def plot_video_metrics(input_folder: str, title_prefix: str = ""):

    paths = {os.path.splitext(f)[0]: os.path.join(input_folder, f)
             for f in os.listdir(input_folder) if f.endswith(".csv")}
    if not paths:
        raise FileNotFoundError("No .csv files found in folder.")

    dfs = {}
    for label, fp in paths.items():
        df = pd.read_csv(fp)
        df['video_ts'] = pd.to_numeric(df['video_ts'], errors='coerce')
        dfs[label] = df
    xmax = max(df['video_ts'].max() for df in dfs.values())

    linestyles = ['--', '-', '-.', ':', (0, (3, 1, 1, 1)), (0, (1, 1))]

    with plt.rc_context(BASE_RC):

        def _plot(ycol, ylabel, title, fname_stub):
            fig, ax = plt.subplots()
            fig.set_size_inches(5, 3)

            for i, (label, df) in enumerate(dfs.items()):
                if ycol not in df.columns:
                    continue
                ax.plot(
                    df['video_ts'], df[ycol],
                    label=label,
                    linestyle=linestyles[i % len(linestyles)]
                )

            _common_format(ax, "Time (s)", ylabel, xmax)

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            ax.legend(
                loc="upper left",
                frameon=False,
                fontsize=7,
                handlelength=1.5,
                borderpad=0.3
            )

            plt.tight_layout()
            pdf_path = os.path.join(OUTPUT_PDF_DIR, f"{fname_stub}.pdf")
            fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
            print(f"Saved → {pdf_path}")
            plt.show()


        _plot('buffer',        'Buffer (s)',
      "Buffer Size Over Time",            "buffer_size")
        _plot('cum_rebuffer',  'Cumulative Rebuffer (s)',
      "Cumulative Rebuffering Over Time", "cum_rebuffer")
        _plot('delivery_rate', 'Delivery Rate (bytes/s)',
      "Delivery Rate Over Time",          "delivery_rate")
        _plot('ssim_index',    'SSIM',
      "Video Quality (SSIM)",             "ssim")
        _plot('rtt',           'RTT (s)',
      "RTT Over Time",                    "rtt")


In [ ]:
plot_video_metrics(
    r"", # Path for the csv files
    title_prefix="" # Title
)
